[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Pipeline Mode &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the `piped` table. Run it first. Each task empties the
table before it starts, so they can be run in any order.


In [1]:
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

STATEMENTS = 5000


def fresh():
    """An empty table, so each timing starts from the same place."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS piped")
        conn.execute("CREATE TABLE piped (id int PRIMARY KEY, note text)")


def rows():
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute("SELECT count(*) FROM piped").fetchone()[0]


def timed(work):
    fresh()
    start = time.perf_counter()
    work()
    return time.perf_counter() - start


def against(baseline, measured):
    """How much faster, as a band, because a timing is not repeatable to a digit."""
    ratio = baseline / measured
    if ratio < 1.2:
        return "no faster"
    if ratio < 3:
        return "somewhat faster"
    return "much faster"


print("server:", start_server())
print(report())
print("pipeline mode available:", psycopg.capabilities.has_pipeline())
fresh()


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
pipeline mode available: True


**1.** Whether the feature is there at all.


In [2]:
print("pipeline mode available:", psycopg.capabilities.has_pipeline())
print("it needs libpq 14 or newer, and psycopg[binary] brings its own")


pipeline mode available: True
it needs libpq 14 or newer, and psycopg[binary] brings its own


Worth asking rather than assuming, because this depends on the libpq the driver was built against
rather than on the server. A wheel with its own libpq answers `True` even against an old server.


**2.** Five statements, queued.


In [3]:
fresh()

with psycopg.connect("dbname=guide") as conn:
    with conn.pipeline():
        for number in range(5):
            conn.execute("INSERT INTO piped VALUES (%s, %s)", (number, "queued"))

print("rows after the block:", rows())


rows after the block: 5


Each `execute` returned before the server had answered, and leaving the block is what made sure all
five had been sent and all five answers read.


**3.** Three answers, three cursors.


In [4]:
with psycopg.connect("dbname=guide") as conn:
    with conn.pipeline():
        cursors = [conn.cursor() for _ in range(3)]
        for number, cur in enumerate(cursors, start=1):
            cur.execute("SELECT %s * %s", (number, 100))

        print("answers:", [cur.fetchone()[0] for cur in cursors])


answers: [100, 200, 300]


One cursor per statement, because a cursor holds one result. Reusing a single cursor here would
quietly give the last statement's answer to whoever asked for the first.


**4.** A failure in the middle.


In [5]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    with conn.pipeline():
        sent = []
        for sql in ("SELECT 10", "SELECT 1/0", "SELECT 30"):
            cur = conn.cursor()
            cur.execute(sql)
            sent.append((sql, cur))

        for sql, cur in sent:
            try:
                print(f"  {sql:<12} -> {cur.fetchone()}")
            except psycopg.Error as error:
                print(f"  {sql:<12} -> {type(error).__name__}: {error}")


  SELECT 10    -> DivisionByZero: division by zero
  SELECT 1/0   -> ProgrammingError: no result available
  SELECT 30    -> ProgrammingError: no result available


`SELECT 10` is the one that raised, and it is the one statement on that list that could not fail.
The division error came back while the code was already past all three, so it surfaced at the first
result anybody asked for.


**5.** What a pipeline will not carry.


In [6]:
with psycopg.connect("dbname=guide") as conn:
    try:
        with conn.pipeline():
            with conn.cursor() as cur:
                with cur.copy("COPY piped FROM STDIN") as copy:
                    copy.write_row((99, "x"))
    except psycopg.NotSupportedError as error:
        print("class:  ", type(error).__module__ + "." + type(error).__name__)
        print("message:", error)


class:   psycopg.NotSupportedError
message: COPY cannot be used in pipeline mode


The class lives in the top-level `psycopg` module rather than in `psycopg.errors`, because
`NotSupportedError` is one of the DB-API's own base classes. A `COPY` takes the connection over while
rows stream, and a pipeline is a queue sharing that connection, so the two cannot both be true.


**6.** What it buys, on this machine.


In [7]:
def plain():
    with psycopg.connect("dbname=guide") as conn:
        for number in range(1000):
            conn.execute("INSERT INTO piped VALUES (%s, %s)", (number, "x"))


def piped():
    with psycopg.connect("dbname=guide") as conn:
        with conn.pipeline():
            for number in range(1000):
                conn.execute("INSERT INTO piped VALUES (%s, %s)", (number, "x"))


baseline = timed(plain)
print("1000 statements over a local socket")
print("  one at a time:", "the baseline")
print("  in a pipeline:", against(baseline, timed(piped)))


1000 statements over a local socket
  one at a time: the baseline
  in a pipeline: somewhat faster


A modest answer, and the right one for where it was measured. A pipeline removes waiting, a local
socket has almost none, and the same comparison against a server a millisecond away would be a
different notebook.


---

&#8592; **Back to:** [Pipeline Mode](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/09-pipeline-mode.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
